## Dataset

This project uses the dataset from the Kaggle competition:

🔗 https://www.kaggle.com/competitions/rsna-intracranial-aneurysm-detection

**The dataset is not included in this repository** due to competition rules.  
Please download it directly from Kaggle after accepting the competition terms.

You can download the data using Kaggle CLI:


kaggle competitions download -c rsna-intracranial-aneurysm-detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from core.datasets import MedicalDataset
from core.evaluation import evaluate_model
from core.preprocessing import train_transform
from core.training import  train_model
from core.helpers import repeat_samples, to_tensor
from core.models import AneurysmClassifier, AneurysmClassifierDINO

In [ ]:
def visualize_stn(model, input_img, device, channels=range(10)):
    model.stn.read_theta = True
    model.eval()
    with torch.no_grad():
        input_img = input_img.to(device)
        stn_output = model.stn(input_img)

        input_img = input_img.cpu().numpy().squeeze(0)
        stn_output = stn_output.cpu().numpy().squeeze(0)

    fig, axes = plt.subplots(2, len(channels), figsize=(len(channels) * 5, 10))
        
    for i, channel in enumerate(channels):
        cmap = 'gray' if channel < 5 else None

        axes[0, i].imshow(input_img[channel, :, :], cmap=cmap)
        axes[1, i].imshow(stn_output[channel, :, :], cmap=cmap)

        axes[0, i].axis('off')
        axes[1, i].axis('off')

        axes[0, i].set_title(f'Original: {channel}')
        axes[1, i].set_title(f'STN: {channel}')

    model.stn.read_theta = False
    plt.show()

In [ ]:
train_folder = Path('C:/Users/dawid/OneDrive/Pulpit/aneurysm classifier/data/train')
test_folder = Path('C:/Users/dawid/OneDrive/Pulpit/aneurysm classifier/data/test')

In [ ]:
x_train = np.load(os.path.join(train_folder, 'train_images.npy'))
y_train = np.load(os.path.join(train_folder, 'train_labels.npy'))
x_test = np.load(os.path.join(test_folder, 'test_images.npy'))
y_test = np.load(os.path.join(test_folder, 'test_labels.npy'))

In [ ]:
bin_x_train = np.copy(x_train)
bin_y_train = np.copy(y_train)
bin_y_test = np.copy(y_test)

aneurysm_train_mask = np.where(y_train > 0)[0]
aneurysm_test_mask = np.where(y_test > 0)[0]
bin_y_train[aneurysm_train_mask] = 1
bin_y_test[aneurysm_test_mask] = 1

In [ ]:
x_train, y_train = repeat_samples(x_train, y_train, 10)

In [ ]:
np.bincount(y_train.squeeze().astype(int))

In [ ]:
np.bincount(bin_y_train[:, 0].astype(int))

In [ ]:
x_train_tensor, y_train_tensor, x_test_tensor, y_test_tensor = to_tensor(x_train, y_train, x_test, y_test)

In [ ]:
train_dataset = MedicalDataset(x_train_tensor, y_train_tensor, train_transform)
test_dataset = MedicalDataset(x_test_tensor, y_test_tensor)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
bin_x_train_tensor, bin_y_train_tensor, bin_x_test_tensor, bin_y_test_tensor = to_tensor(
    bin_x_train, bin_y_train, x_test, bin_y_test)

bin_train_dataset = MedicalDataset(bin_x_train_tensor, bin_y_train_tensor, train_transform)
bin_test_dataset = MedicalDataset(bin_x_test_tensor, bin_y_test_tensor)

batch_size = 32

bin_train_loader = DataLoader(bin_train_dataset, batch_size, shuffle=True)
bin_test_loader = DataLoader(bin_test_dataset, batch_size, shuffle=False)

BINARY MODEL

In [ ]:
num_epochs = 8
lr = 1e-4

bin_model = AneurysmClassifier(in_channels=10, num_classes=2)
bin_model.to(device)

'''stn_params = [p for p in bin_model.stn.parameters() if p.requires_grad]
resnet_params = [p for p in bin_model.resnet.parameters() if p.requires_grad]'''

params = [p for p in bin_model.parameters() if p.requires_grad]

bin_optimizer = torch.optim.Adam(params=params, lr=lr)

class_weights = torch.tensor([y_train.shape[0] / count for count in np.bincount(bin_y_train_tensor[:,0])], dtype=torch.float32, device=device)
#class_weights[1] *= 1.1
bin_criterion = nn.CrossEntropyLoss(weight=class_weights)

BINARY CLASSIFICATION

In [ ]:
best_model_weights, history = train_model(
    bin_model, bin_optimizer,bin_criterion, bin_train_loader, bin_test_loader, num_epochs, device)

BINARY MODEL EVALUATION

In [ ]:
bin_model.load_state_dict(best_model_weights)

In [ ]:
evaluate_model(bin_model, device, bin_test_loader)

In [ ]:
def get_idx():
    idx = -1
    while True:
        idx += 1
        yield idx

idx_gen = get_idx()
bin_idx_gen = get_idx()
bin_dino_idx = get_idx()

In [ ]:
bin_train_loader.dataset.transform = None

for input, _ in bin_train_loader:
    visualize_stn(bin_model, input[0].unsqueeze(0), device,)
    break

In [ ]:
bin_idx = next(bin_idx_gen)
print(bin_idx)
visualize_stn(bin_model, input[bin_idx].unsqueeze(0), device)

In [ ]:
visualize_stn(bin_model, input[bin_idx].unsqueeze(0), device, (2, 7))

MULTICLASS CLASSIFICATION MODEL

In [ ]:
num_epochs = 8
resnet_lr = 1e-5
stn_lr = 1e-5

model = AneurysmClassifier(in_channels=10, num_classes=14)
model.to(device)

stn_params = [p for p in model.stn.parameters() if p.requires_grad]
resnet_params = [p for p in model.resnet.parameters() if p.requires_grad]

optimizer = torch.optim.Adam([
    {'params' : stn_params, 'lr' : stn_lr}, 
    {'params' : resnet_params, 'lr' : resnet_lr},
    ])

class_weights = torch.tensor([y_train.shape[0] / count for count in np.bincount(y_train_tensor[:,0])], dtype=torch.float32, device=device)
#class_weights[1:] *= 1.1
criterion = nn.CrossEntropyLoss(weight=class_weights)

MULTICLASS CLASSIFICATION

In [ ]:
best_model_weights, history = train_model(
    model, optimizer, criterion, train_loader, test_loader, num_epochs, device)

In [ ]:
model.load_state_dict(best_model_weights)

In [ ]:
torch.save(model.state_dict(), 'multiclass__resnet50.pth')

In [ ]:
evaluate_model(model, device, test_loader)

In [ ]:
for input, label in train_loader:
    visualize_stn(model, input[0].unsqueeze(0), device,)
    break

In [ ]:
idx = next(idx_gen)
print(idx)
visualize_stn(model, input[idx].unsqueeze(0), device)

In [ ]:
visualize_stn(model, input[idx].unsqueeze(0), device, (2, 7))
label[idx]

BINARY DINO

In [ ]:
num_epochs = 8
lr = 1e-5

bin_dino = AneurysmClassifierDINO(in_channels=10, num_classes=2)
bin_dino.to(device)

params = [p for p in bin_dino.parameters() if p.requires_grad]

bin_dino_optimizer = torch.optim.AdamW(params, lr)

class_weights = torch.tensor([y_train.shape[0] / count for count in np.bincount(bin_y_train_tensor[:,0])], dtype=torch.float32, device=device)
#class_weights[1] *= 1.1
bin_dino_criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
best_model_weights, history = train_model(
    bin_dino, bin_dino_optimizer, bin_dino_criterion, bin_train_loader, bin_test_loader, num_epochs, device)

EVALUATE BINARY DINO

In [ ]:
bin_dino.load_state_dict(best_model_weights)

In [ ]:
torch.save(bin_dino.state_dict(), 'bin_dino.pth')

In [ ]:
evaluate_model(bin_dino, device, bin_test_loader)

In [ ]:
bin_train_loader.dataset.transform = None

for input, label in bin_train_loader:
    visualize_stn(bin_dino, input[0].unsqueeze(0), device,)
    break

In [ ]:

idx = next(bin_dino_idx)
print(idx)
visualize_stn(bin_dino, input[idx].unsqueeze(0), device)